# Vessel Segmentation — SegFormer (Run 7)
Standalone Kaggle notebook. One new run: **SegFormer (MiT-b0 encoder), augmentation ON**, per the professor's requested architecture comparison, alongside U-Net++, Attention U-Net, and DeepLabV3+.

**Mistakes from earlier runs, fixed proactively here (not discovered the hard way again):**
1. **Background color convention** — `Training/` masks use black background, `Testing/` masks use white. This notebook's color-matching treats both as background from the start (see Cell 6), instead of assuming one convention like the first DeepLabV3+/vessel attempt did.
2. **Missing/wrong input normalization** — uses the exact same ImageNet mean/std as every other run (Cell 7's `A.Normalize`), so results stay comparable and the encoder doesn't see out-of-distribution inputs like the first inference attempt did.
3. **SegFormer-specific output shape** — SegFormer's decode head outputs logits at 1/4 input resolution by design. This is handled inside `SegformerWrapper` (Cell 11) so every downstream cell (loss, metrics, training loop) just sees full-resolution logits like every other model — with an explicit shape assertion (Cell 12) that fails loudly *before* a full 18-epoch run if this ever breaks, instead of surfacing as a confusing shape-mismatch deep into training.
4. **Checkpoint reload without guessing architecture** — saves via both a raw `state_dict` (for consistency with your other checkpoints) *and* `save_pretrained()` (Cell 16), so a future inference notebook can reload the exact architecture + weights without anyone having to reverse-engineer layer names from a state_dict error, the way Attention U-Net's checkpoint had to be earlier.
5. **File pairing by numeric ID, not position** — Cell 5, same explicit ID-matching used after the vessel test-set pairing issue.
6. **Appropriate optimizer/LR for a transformer** — AdamW with a much lower LR (6e-5) than the CNN runs' 5e-4 (Cell 14) — using the CNN learning rate on a pretrained transformer backbone risks destroying the pretrained weights in the first few steps.

**Before running:**
1. Upload your vessel dataset as a Kaggle Dataset, add via "+ Add Input".
2. Settings → Accelerator → GPU T4x2 or P100 → Internet ON.
3. Settings → Persistence → "Files only" (protects `/kaggle/working` if the session disconnects overnight).
4. Edit `KAGGLE_INPUT_DIR` in Cell 2 to your actual dataset slug.

## Cell 1 — Install dependencies

In [1]:
!apt-get install -y p7zip-full -qq
!pip install -q transformers albumentations tqdm

## Cell 2 — Locate + extract dataset
**Edit `KAGGLE_INPUT_DIR` below** to match your dataset slug (check the "Input" panel, or run `!find /kaggle/input -maxdepth 4` first if unsure — don't assume the flat `/kaggle/input/<slug>/` layout, per the earlier Kaggle path lesson).

In [2]:
import os

!find /kaggle/input -maxdepth 4

# TODO: replace with your actual dataset slug (from the find output above)
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/zakariaassadiki/vessel-dataset"
LOCAL_ROOT = "/kaggle/working/vessel_data"
os.makedirs(LOCAL_ROOT, exist_ok=True)

VESSEL_7Z_PATH = os.path.join(KAGGLE_INPUT_DIR, "vessel_dataset.7z")  # adjust filename if different
if os.path.exists(VESSEL_7Z_PATH):
    !7z x "{VESSEL_7Z_PATH}" -o{LOCAL_ROOT} -y
else:
    LOCAL_ROOT = KAGGLE_INPUT_DIR
    print(f"No .7z found -- assuming Kaggle already extracted your zip. Using LOCAL_ROOT = {LOCAL_ROOT}")

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/zakariaassadiki
/kaggle/input/datasets/zakariaassadiki/vessel-dataset
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset
No .7z found -- assuming Kaggle already extracted your zip. Using LOCAL_ROOT = /kaggle/input/datasets/zakariaassadiki/vessel-dataset


## Cell 3 — Confirm real folder structure (don't assume it)

In [3]:
DATASET_ROOT = os.path.join(LOCAL_ROOT, "VascularModelDataset")
TRAIN_PATCH_DIR = os.path.join(DATASET_ROOT, "Training", "patches")
TRAIN_MASK_DIR  = os.path.join(DATASET_ROOT, "Training", "masks")
TEST_PATCH_DIR  = os.path.join(DATASET_ROOT, "Testing", "testset_images")
TEST_MASK_DIR   = os.path.join(DATASET_ROOT, "Testing", "testset_masks")

import glob
for name, d in [("train patches", TRAIN_PATCH_DIR), ("train masks", TRAIN_MASK_DIR),
                ("test patches", TEST_PATCH_DIR), ("test masks", TEST_MASK_DIR)]:
    n = len(glob.glob(os.path.join(d, "*.tiff")))
    print(f"{name:14s}: {n} files  ({d})")

train patches : 6544 files  (/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/patches)
train masks   : 6544 files  (/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/masks)
test patches  : 728 files  (/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing/testset_images)
test masks    : 728 files  (/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing/testset_masks)


## Cell 4 — Pair files by numeric ID (not by position)
A single gap in either folder's numbering silently shifts every pair after it if you just sort-and-zip two lists — pair explicitly by the numeric ID in the filename instead.

In [4]:
import re

def numeric_id(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    return int("".join(ch for ch in stem if ch.isdigit()))

def pair_by_id(images_dir, masks_dir, img_prefix="patch", mask_prefix="mask"):
    img_by_id = {numeric_id(p): p for p in glob.glob(os.path.join(images_dir, f"{img_prefix}_*.tiff"))}
    mask_by_id = {numeric_id(p): p for p in glob.glob(os.path.join(masks_dir, f"{mask_prefix}_*.tiff"))}
    common = sorted(set(img_by_id) & set(mask_by_id))
    only_img = set(img_by_id) - set(mask_by_id)
    only_mask = set(mask_by_id) - set(img_by_id)
    if only_img or only_mask:
        print(f"  WARNING: {len(only_img)} images with no mask, {len(only_mask)} masks with no image")
    return [(img_by_id[i], mask_by_id[i]) for i in common]

training_pairs = pair_by_id(TRAIN_PATCH_DIR, TRAIN_MASK_DIR)
test_pairs = pair_by_id(TEST_PATCH_DIR, TEST_MASK_DIR)

print(f"Training pairs: {len(training_pairs)}  (expect ~6544)")
print(f"Test pairs: {len(test_pairs)}  (expect ~728)")

Training pairs: 6544  (expect ~6544)
Test pairs: 728  (expect ~728)


## Cell 5 — Train/val split (random, fixed seed, matches Runs 1-6)

In [5]:
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(training_pairs, test_size=0.2, random_state=42)
print(f"Train: {len(train_pairs)}  Val: {len(val_pairs)}")

Train: 5235  Val: 1309


## Cell 6 — Constants + background-convention-agnostic color matching
**Fixes the bug that cost real debugging time on the DeepLabV3+ run:** `Training/` masks encode background as black `(0,0,0)`, `Testing/` masks encode it as white `(255,255,255)`. Matching against only one of them silently misclassifies every background pixel of the *other* convention as `bile_duct` — purple happens to be white's nearest neighbor among the 5 pure class colors. Both are treated as background from the start here, for both training and later inference on this checkpoint.

In [6]:
import numpy as np

NUM_CLASSES = 5
PATCH_SIZE = 256
CLASS_NAMES = ["background", "portal_vein", "central_vein", "artery", "bile_duct"]

# Extended reference set: BOTH black and white map to class 0 (background).
# Portal vein / central vein / artery / bile duct pure colors are unaffected.
_MATCH_COLORS = np.array([
    [0,   0,   0],    # -> class 0 (background, Training/ convention)
    [255, 255, 255],  # -> class 0 (background, Testing/ convention)
    [0,   0, 255],    # -> class 1 (portal vein)
    [0, 255,   0],    # -> class 2 (central vein)
    [255,  0,   0],   # -> class 3 (artery)
    [128,  0, 128],   # -> class 4 (bile duct)
], dtype=np.int32)
_MATCH_COLOR_TO_CLASS = np.array([0, 0, 1, 2, 3, 4])

# Display-only palette (used when rendering predictions/GT as images later)
DISPLAY_COLORS = np.array([
    [0, 0, 0], [0, 0, 255], [0, 255, 0], [255, 0, 0], [128, 0, 128],
], dtype=np.uint8)

UM_PER_PX = 0.227 * 4  # native 0.227 um/px, 4x downsample -> ~0.908 um/px

## Cell 7 — Dataset class
Resize FIRST (image: area interp, mask: nearest), THEN color-match on the small array — doing it at full 1024x1024 resolution first was the original ~25-30 min/epoch bottleneck on Run 1. Same ImageNet normalization as every other run.

In [7]:
import cv2
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_train_transforms(aug_on: bool):
    if aug_on:
        return A.Compose([
            A.Rotate(limit=180, border_mode=cv2.BORDER_REFLECT, p=0.7),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ElasticTransform(alpha=1, sigma=20, p=0.3),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.4),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_eval_transforms():
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def rgb_mask_to_class_index(mask_rgb):
    h, w, _ = mask_rgb.shape
    flat = mask_rgb.reshape(-1, 3).astype(np.int32)
    dists = np.sum((flat[:, None, :] - _MATCH_COLORS[None, :, :]) ** 2, axis=2)
    nearest = np.argmin(dists, axis=1)
    return _MATCH_COLOR_TO_CLASS[nearest].reshape(h, w).astype(np.int64)

class VesselDataset(Dataset):
    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)

        mask_bgr = cv2.imread(mask_path, cv2.IMREAD_COLOR)
        mask_rgb = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2RGB)
        mask_rgb = cv2.resize(mask_rgb, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = rgb_mask_to_class_index(mask_rgb)

        transformed = self.transform(image=image, mask=mask)
        return transformed["image"], transformed["mask"].long()

## Cell 8 — Loss: Dice + Focal (same as every other run)

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_onehot, dims)
        cardinality = torch.sum(probs + targets_onehot, dims)
        dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

class DiceFocalLoss(nn.Module):
    def __init__(self, num_classes, dice_weight=0.5, focal_weight=0.5):
        super().__init__()
        self.dice = DiceLoss(num_classes)
        self.focal = FocalLoss()
        self.dw, self.fw = dice_weight, focal_weight

    def forward(self, logits, targets):
        return self.dw * self.dice(logits, targets) + self.fw * self.focal(logits, targets)

## Cell 9 — Per-class Dice metric

In [9]:
def per_class_dice(logits, targets, num_classes, smooth=1e-6):
    preds = torch.argmax(logits, dim=1)
    dices = []
    for c in range(num_classes):
        pred_c = (preds == c).float()
        target_c = (targets == c).float()
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        dices.append(((2. * intersection + smooth) / (union + smooth)).item())
    return dices

## Cell 10 — SegFormer model
`nvidia/mit-b0`: the SegFormer encoder pretrained on ImageNet, without an ADE20k-specific 150-class head to override — the more natural fine-tuning starting point for a custom 5-class task, and avoids a noisy "mismatched head size, reinitializing" warning on every load.

In [10]:
from transformers import SegformerForSemanticSegmentation

SEGFORMER_CHECKPOINT = "nvidia/mit-b0"

def build_segformer(num_classes=NUM_CLASSES):
    return SegformerForSemanticSegmentation.from_pretrained(
        SEGFORMER_CHECKPOINT,
        num_labels=num_classes,
    )

## Cell 11 — Wrapper: upsample SegFormer's logits back to input resolution
**This is the specific SegFormer pitfall to avoid.** SegFormer's decode head outputs logits at 1/4 of input resolution by design (e.g. 256x256 input -> 64x64 logits). Feeding that straight into the same loss/metric functions used for the other runs would either crash on a shape mismatch or silently compute loss against the wrong spatial scale. Wrapping the model so its `forward()` always returns full-resolution logits means every downstream cell (loss, metrics, training loop) can treat it exactly like U-Net++, DeepLabV3+, or Attention U-Net -- no special-casing needed anywhere else.

In [11]:
class SegformerWrapper(nn.Module):
    def __init__(self, hf_model, target_size):
        super().__init__()
        self.hf_model = hf_model
        self.target_size = target_size  # (H, W)

    def forward(self, pixel_values):
        logits = self.hf_model(pixel_values=pixel_values).logits  # (N, C, H/4, W/4)
        logits = F.interpolate(logits, size=self.target_size, mode="bilinear", align_corners=False)
        return logits

def build_model(architecture: str, num_classes: int = NUM_CLASSES):
    if architecture == "segformer":
        return SegformerWrapper(build_segformer(num_classes), target_size=(PATCH_SIZE, PATCH_SIZE))
    raise ValueError(f"Unknown architecture: {architecture}")

## Cell 12 — Sanity check: verify output shape BEFORE training
Catches a shape problem in seconds instead of discovering it deep into an 18-epoch run. Fails loudly here rather than silently misbehaving later.

In [12]:
_sanity_model = build_model("segformer").to("cuda" if torch.cuda.is_available() else "cpu")
_device = next(_sanity_model.parameters()).device
_dummy_input = torch.randn(2, 3, PATCH_SIZE, PATCH_SIZE).to(_device)
with torch.no_grad():
    _dummy_output = _sanity_model(_dummy_input)

expected_shape = (2, NUM_CLASSES, PATCH_SIZE, PATCH_SIZE)
assert _dummy_output.shape == expected_shape, (
    f"Output shape {tuple(_dummy_output.shape)} != expected {expected_shape} -- "
    f"the upsampling in SegformerWrapper isn't matching PATCH_SIZE. Fix before training."
)
print(f"Output shape OK: {tuple(_dummy_output.shape)}")
del _sanity_model, _dummy_input, _dummy_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

model.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

Output shape OK: (2, 5, 256, 256)


## Cell 13 — Training loop
Same structure as Runs 1-6's `train_run`. `num_workers=2` + `persistent_workers=True` + `prefetch_factor=2` from the start — the earlier `num_workers=0` "fix" for a misdiagnosed deadlock was actually a severe regression.

In [13]:
import time
import pandas as pd
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

def train_run(run_name, architecture, aug_on, epochs, batch_size, lr, weight_decay=0.01, max_batches=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"=== {run_name} | arch={architecture} aug_on={aug_on} epochs={epochs} bs={batch_size} lr={lr} | device={device} ===")

    train_ds = VesselDataset(train_pairs, get_train_transforms(aug_on))
    val_ds = VesselDataset(val_pairs, get_eval_transforms())

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True,
                               persistent_workers=True, prefetch_factor=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=True,
                             persistent_workers=True, prefetch_factor=2)

    model = build_model(architecture).to(device)
    criterion = DiceFocalLoss(NUM_CLASSES)
    # AdamW + a much lower LR than the CNN runs (5e-4) -- a pretrained transformer backbone
    # can have its pretrained weights wrecked in a handful of steps at CNN-scale learning rates.
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()

    checkpoint_dir = "/kaggle/working/checkpoints"
    log_dir = "/kaggle/working/logs"
    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    log_path = f"{log_dir}/{run_name}.csv"

    log_rows = []
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_bar = tqdm(train_loader, desc=f"[{run_name}] epoch {epoch+1}/{epochs} train", leave=False)
        for batch_idx, (images, masks) in enumerate(train_bar):
            if max_batches is not None and batch_idx >= max_batches:
                break
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")
        n_train_batches = min(len(train_loader), max_batches) if max_batches else len(train_loader)
        train_loss /= n_train_batches

        model.eval()
        val_loss = 0.0
        class_dice_sums = [0.0] * NUM_CLASSES
        n_batches = 0
        val_bar = tqdm(val_loader, desc=f"[{run_name}] epoch {epoch+1}/{epochs} val", leave=False)
        with torch.no_grad():
            for batch_idx, (images, masks) in enumerate(val_bar):
                if max_batches is not None and batch_idx >= max_batches:
                    break
                images, masks = images.to(device), masks.to(device)
                with autocast():
                    logits = model(images)
                    loss = criterion(logits, masks)
                val_loss += loss.item()
                dices = per_class_dice(logits, masks, NUM_CLASSES)
                for c in range(NUM_CLASSES):
                    class_dice_sums[c] += dices[c]
                n_batches += 1
        val_loss /= n_batches
        class_dice_avg = [s / n_batches for s in class_dice_sums]

        scheduler.step()

        row = {"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
               "lr": scheduler.get_last_lr()[0]}
        for c, name in enumerate(CLASS_NAMES):
            row[f"dice_{name}"] = class_dice_avg[c]
        log_rows.append(row)

        print(f"epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | "
              f"dice(bg/pv/cv/art/bd) = {[round(d,3) for d in class_dice_avg]}")

        if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
            torch.save(model.state_dict(), f"{checkpoint_dir}/{run_name}.pth")

        pd.DataFrame(log_rows).to_csv(log_path, index=False)

    total_time = time.time() - start_time
    print(f"=== {run_name} done in {total_time/60:.1f} min ===")

    return model, {
        "run_name": run_name, "architecture": architecture, "aug_on": aug_on,
        "epochs": epochs, "batch_size": batch_size, "lr": lr,
        "final_val_loss": val_loss,
        "final_dice_overall": sum(class_dice_avg[1:]) / (NUM_CLASSES - 1),
        **{f"final_dice_{name}": class_dice_avg[c] for c, name in enumerate(CLASS_NAMES)},
        "train_time_min": total_time / 60,
    }

## Run 7 — SegFormer (MiT-b0), augmentation ON
Same loss (Dice+Focal), same 18 epochs / 256×256 as every other run, so it's directly comparable in the final table. SegFormer-B0 is lightweight (~3.7M params vs ResNet34's ~21M) so this often trains *faster* per epoch than the CNN runs, even with the extra upsampling step. Drop `batch_size` to 8 if you OOM.

In [14]:
trained_model, result_run7 = train_run(
    run_name="vessel_run7_segformer_augON",
    architecture="segformer",
    aug_on=True,
    epochs=18,
    batch_size=16,   # drop to 8 if this OOMs
    lr=6e-5,         # AdamW, transformer-appropriate LR -- NOT the CNN runs' 5e-4
)

=== vessel_run7_segformer_augON | arch=segformer aug_on=True epochs=18 bs=16 lr=6e-05 | device=cuda ===


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

[vessel_run7_segformer_augON] epoch 1/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

/tmp/ipykernel_58/4247011678.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

[vessel_run7_segformer_augON] epoch 17/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 17/18 | train_loss 0.1565 | val_loss 0.1624 | dice(bg/pv/cv/art/bd) = [0.982, 0.795, 0.774, 0.722, 0.593]


[vessel_run7_segformer_augON] epoch 18/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

[vessel_run7_segformer_augON] epoch 18/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 18/18 | train_loss 0.1581 | val_loss 0.1626 | dice(bg/pv/cv/art/bd) = [0.982, 0.794, 0.773, 0.72, 0.592]
=== vessel_run7_segformer_augON done in 58.0 min ===


## Save checkpoint two ways — avoids the "guess the architecture" problem
1. Raw `state_dict` (`.pth`) — consistent with your other checkpoints, for any script that already expects that format.
2. `save_pretrained()` — saves the HF model's config + weights together, so it reloads with `SegformerForSemanticSegmentation.from_pretrained(path)` with zero architecture guessing later. This is exactly the step that was missing for Attention U-Net, which cost real debugging time reverse-engineering its layer names from a failed `load_state_dict`.

In [15]:
hf_save_dir = "/kaggle/working/checkpoints/vessel_run7_segformer_hf"
os.makedirs(hf_save_dir, exist_ok=True)
trained_model.hf_model.save_pretrained(hf_save_dir)
print(f"Saved HF format to {hf_save_dir} -- reload with:")
print(f'  SegformerForSemanticSegmentation.from_pretrained("{hf_save_dir}")')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved HF format to /kaggle/working/checkpoints/vessel_run7_segformer_hf -- reload with:
  SegformerForSemanticSegmentation.from_pretrained("/kaggle/working/checkpoints/vessel_run7_segformer_hf")


## Save run log

In [16]:
runs_log_run7 = pd.DataFrame([result_run7])
runs_log_path = "/kaggle/working/runs_log_vessel_run7.csv"
runs_log_run7.to_csv(runs_log_path, index=False)
print(f"Saved to {runs_log_path}")
runs_log_run7

Saved to /kaggle/working/runs_log_vessel_run7.csv


,run_name,architecture,aug_on,epochs,batch_size,lr,final_val_loss,final_dice_overall,final_dice_background,final_dice_portal_vein,final_dice_central_vein,final_dice_artery,final_dice_bile_duct,train_time_min
0,vessel_run7_segformer_augON,segformer,True,18,16,0.00006,0.162637,0.719686,0.981726,0.794266,0.772926,0.719807,0.591743,57.951524


## Zip + download
No direct Kaggle↔Drive integration — download from the Output panel, then merge into your shared Drive folder alongside Runs 1-6. If you're about to close the tab or sleep, confirm Settings → Persistence is "Files only" (not "No persistence") first, or use **Save Version → Save & Run All (Commit)** so this runs unattended and its outputs persist regardless of your session/browser.

In [17]:
!zip -r /kaggle/working/vessel_run7_results.zip /kaggle/working/checkpoints /kaggle/working/logs /kaggle/working/runs_log_vessel_run7.csv

  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/vessel_run7_segformer_hf/ (stored 0%)
  adding: kaggle/working/checkpoints/vessel_run7_segformer_hf/config.json (deflated 59%)
  adding: kaggle/working/checkpoints/vessel_run7_segformer_hf/model.safetensors (deflated 7%)
  adding: kaggle/working/checkpoints/vessel_run7_segformer_augON.pth (deflated 8%)
  adding: kaggle/working/logs/ (stored 0%)
  adding: kaggle/working/logs/vessel_run7_segformer_augON.csv (deflated 50%)
  adding: kaggle/working/runs_log_vessel_run7.csv (deflated 39%)
